# 武汉测量手动平均实验 (1800 / 3300 / 5900 MHz)

**目标**：对比两种手动平均方式在不同平均数目下的 PDP 表现。

- **实验 1**：先在 IQ 域做复数平均，再做匹配滤波 → CIR（相干平均）
- **实验 2**：把 15 个 IQ 拼接成长序列做滑动相关，得到 15 个 CIR 后再平均（先相关后平均）

两个实验均跳过第 1 个 IQ，对以下子集取平均：`#2`, `#2-3`, `#2-5`, `#2-13`, `#2-15`

仅分析每个频点**第一个 snapshot（frame 0）**。

## Section 1: Setup & Imports

In [1]:
import sys
import math
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io.bin_reader_luoyang import LuoyangBinReader
from src.io.bin_read import _S_MATCHED, BW_HZ, U, _N_FFT

matplotlib.rcParams.update({'font.family': 'Times New Roman', 'font.size': 12})
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root : {PROJECT_ROOT}')
print(f'Device       : {DEVICE}')
print(f'U={U}, BW={BW_HZ/1e6:.0f} MHz, _N_FFT={_N_FFT}')

ModuleNotFoundError: No module named 'torch'

## Section 2: Load Wuhan Data

In [ ]:
DATA_DIR = Path('/mnt/win_data/data_mea/data_save/Mea_data/20250902_wuhan_1800_3300_5900_move')

FREQ_FILES = {
    '1800M': DATA_DIR / '接收数据帧_20250906175626305_1800M.bin',
    '3300M': DATA_DIR / '接收数据帧_20250906174712710_3300M.bin',
    '5900M': DATA_DIR / '接收数据帧_20250906173345013_5900M.bin',
}

reader = LuoyangBinReader()
raw_iq = {}

for freq, path in FREQ_FILES.items():
    iq = reader.read_iq_sequences(path)  # (n_frames, 15, U) complex64
    raw_iq[freq] = iq
    print(f'{freq}: shape={iq.shape}, dtype={iq.dtype},  n_frames={iq.shape[0]}')

## Section 3: Definitions — Subsets & Correlation Functions

In [ ]:
# IQ 子集（0-indexed，跳过第 0 个）
SUBSETS = {
    '#2':    [1],
    '#2-3':  list(range(1, 3)),
    '#2-5':  list(range(1, 5)),
    '#2-13': list(range(1, 13)),
    '#2-15': list(range(1, 15)),
}

delay_ns = np.arange(U) / BW_HZ * 1e9  # 时延轴 (ns)


def correlate_single(iq_seq: np.ndarray) -> np.ndarray:
    """匹配滤波单个 IQ 块 (U,) → CIR (U,).

    与 _sliding_correlate 等价：DC 去除 → 三倍平铺 → 循环相关 → 取中段。
    """
    x = (iq_seq - iq_seq.mean()).astype(np.complex64)
    ext = np.tile(x, 3)
    F = np.fft.fft(ext, n=_N_FFT)
    S_f = np.fft.fft(_S_MATCHED, n=_N_FFT)
    corr = np.fft.ifft(F * S_f)
    return (corr[2 * U - 1 : 3 * U - 1] / U).astype(np.complex64)


def correlate_long(iq_long: np.ndarray) -> np.ndarray:
    """15*U 长序列滑动匹配滤波 → (15, U) CIR 矩阵.

    步骤：
      1. DC 去除
      2. 线性互相关（FFT, n_fft >= 16U-1）
      3. 截取前 16U-1 个线性相关点，去掉前 U-1 个边界点
      4. reshape (15, U)，除以 U 归一化

    输出第 m 行对应第 m+1 个 IQ（0-indexed）的 CIR 估计。
    """
    N = len(iq_long)
    assert N == 15 * U, f'Expected 15*U={15*U}, got {N}'
    # 线性卷积需要 n_fft >= N + U - 1 = 16U-1
    n_fft_long = 1 << math.ceil(math.log2(N + U))  # = 16384

    x = (iq_long - iq_long.mean()).astype(np.complex64)
    X = np.fft.fft(x, n=n_fft_long)
    S_f = np.fft.fft(_S_MATCHED, n=n_fft_long)
    corr = np.fft.ifft(X * S_f)  # 长度 n_fft_long，前 N+U-1 为线性相关结果

    # 去掉前 U-1 边界点，取 15U 个有效点
    valid = corr[U - 1 : U - 1 + N]  # shape (15*U,)
    return (valid.reshape(15, U) / U).astype(np.complex64)


print(f'n_fft_long = {1 << math.ceil(math.log2(15*U + U))} (>= 16U-1={16*U-1})')
print('Subset definitions:')
for k, v in SUBSETS.items():
    print(f'  {k:8s}: idx={v}  (M={len(v)})')

## Section 4: Experiment 1 — IQ 域复数平均 → 匹配滤波

**流程**：选取子集 IQ → 复数均值 → `correlate_single` → CIR

本质：相干平均（coherent averaging），SNR 增益 ∝ M（线性）。
前提：各 IQ 之间相位对齐（发射机 CFO 足够小）。

In [ ]:
SNAP = 0  # 分析第一个 snapshot

exp1_cir = {}  # {freq: {subset_name: cir (U,) complex64}}

for freq, iq_all in raw_iq.items():
    iq_frame = iq_all[SNAP]  # (15, U)
    exp1_cir[freq] = {}
    for name, idx in SUBSETS.items():
        iq_avg = iq_frame[idx, :].mean(axis=0)  # 复数平均 → (U,)
        cir = correlate_single(iq_avg)           # (U,)
        exp1_cir[freq][name] = cir

print('Exp1 done. CIR shapes:')
for freq in exp1_cir:
    shapes = {k: v.shape for k, v in exp1_cir[freq].items()}
    print(f'  {freq}: {shapes}')

## Section 5: Experiment 2 — 长序列滑动相关 → 平均 CIR

**流程**：将 15 个 IQ 拼接成长度 15×1024 的序列 → `correlate_long` → (15, U) CIR 矩阵 → 选取子集取复数均值

本质：先做相关再平均（各 CIR 独立估计），若 IQ 间存在相位漂移，平均时相位不齐会导致 SNR 增益下降。

In [ ]:
exp2_cir = {}  # {freq: {subset_name: cir (U,) complex64}}

for freq, iq_all in raw_iq.items():
    iq_frame = iq_all[SNAP]        # (15, U)
    iq_long = iq_frame.reshape(-1) # (15*U,) 全部 15 个 IQ 拼接

    cirs_all = correlate_long(iq_long)  # (15, U)
    exp2_cir[freq] = {}

    for name, idx in SUBSETS.items():
        cir_avg = cirs_all[idx, :].mean(axis=0)  # 复数平均 CIR → (U,)
        exp2_cir[freq][name] = cir_avg

print('Exp2 done. CIR shapes:')
for freq in exp2_cir:
    shapes = {k: v.shape for k, v in exp2_cir[freq].items()}
    print(f'  {freq}: {shapes}')

## Section 6: 辅助函数 — PDP 绘图

In [ ]:
FREQS = ['1800M', '3300M', '5900M']
SUBSET_NAMES = list(SUBSETS.keys())


def _snr_db(pdp: np.ndarray, peak_bin: int) -> float:
    """Peak-to-noise-floor SNR (dB). Noise floor = median of bins > 50 away from peak."""
    mask = np.abs(np.arange(len(pdp)) - peak_bin) > 50
    noise_floor = np.median(pdp[mask]) if mask.sum() > 10 else pdp.min()
    return float(10 * np.log10(pdp[peak_bin] / max(noise_floor, 1e-30)))


def plot_pdp(cir_dict: dict, suptitle_prefix: str, xlim: tuple = (0, 5000)) -> None:
    """仿 Section 7 格式，每个频点画 1 行 5 列 PDP 子图。"""
    for freq in FREQS:
        fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=True)
        fig.suptitle(f'{suptitle_prefix} — {freq}  (Snapshot {SNAP})',
                     fontsize=12, fontweight='bold')

        for ax, name in zip(axes, SUBSET_NAMES):
            cir = cir_dict[freq][name]
            pdp = np.abs(cir) ** 2
            pdp_db = 10 * np.log10(pdp + 1e-30)
            pdp_db -= pdp_db.max()           # 归一化到 0 dB

            peak_bin = int(np.argmax(pdp))
            peak_delay = peak_bin * 1e9 / BW_HZ
            snr = _snr_db(pdp, peak_bin)

            ax.plot(delay_ns, pdp_db, lw=1)
            ax.axvline(peak_delay, color='r', ls='--', lw=1, alpha=0.8,
                       label=f'{peak_delay:.0f} ns\nSNR={snr:.1f} dB')
            ax.set_xlabel('Delay (ns)', fontsize=10)
            if ax is axes[0]:
                ax.set_ylabel('PDP (dB)', fontsize=10)
            ax.set_title(f'M={len(SUBSETS[name])}  ({name})', fontsize=10)
            ax.set_xlim(xlim)
            ax.set_ylim([-50, 3])
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

        fig.tight_layout()
        plt.show()


print('Plot function defined.')

## Section 7: PDP Comparison — Experiment 1 (IQ 域平均后相关)

In [ ]:
plot_pdp(exp1_cir, 'Exp1: IQ avg → correlate')

## Section 8: PDP Comparison — Experiment 2 (长序列滑动相关后平均 CIR)

In [ ]:
plot_pdp(exp2_cir, 'Exp2: concatenate → correlate → CIR avg')

## Section 9: Exp 1 vs Exp 2 — 同频点同子集叠加对比

蓝色 = Exp1，橙色 = Exp2。观察两种方式在 PDP 形状和噪底上的差异。

In [ ]:
for freq in FREQS:
    fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=True)
    fig.suptitle(f'Exp1 vs Exp2 — {freq}  (Snapshot {SNAP})',
                 fontsize=12, fontweight='bold')

    for ax, name in zip(axes, SUBSET_NAMES):
        for cir_dict, label, color in [
            (exp1_cir, 'Exp1 (IQ avg)', 'tab:blue'),
            (exp2_cir, 'Exp2 (CIR avg)', 'tab:orange'),
        ]:
            cir = cir_dict[freq][name]
            pdp = np.abs(cir) ** 2
            pdp_db = 10 * np.log10(pdp + 1e-30)
            pdp_db -= pdp_db.max()
            ax.plot(delay_ns, pdp_db, lw=1, color=color, label=label, alpha=0.8)

        ax.set_xlabel('Delay (ns)', fontsize=10)
        if ax is axes[0]:
            ax.set_ylabel('PDP (dB)', fontsize=10)
        ax.set_title(f'M={len(SUBSETS[name])}  ({name})', fontsize=10)
        ax.set_xlim([0, 5000])
        ax.set_ylim([-50, 3])
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## Section 10: SNR vs 平均数目汇总表

In [ ]:
header = '{:>8}  {:>8}  {:>4}  {:>14}  {:>14}  {:>10}'.format(
    'Freq', 'Subset', 'N', 'P2N IQ Ave(dB)', 'P2N CIR AVE(dB)', 'Diff(dB)')
print(header)
print('-' * 72)

for freq in FREQS:
    for name, idx in SUBSETS.items():
        m = len(idx)

        cir1 = exp1_cir[freq][name]
        pdp1 = np.abs(cir1) ** 2
        snr1 = _snr_db(pdp1, int(np.argmax(pdp1)))

        cir2 = exp2_cir[freq][name]
        pdp2 = np.abs(cir2) ** 2
        snr2 = _snr_db(pdp2, int(np.argmax(pdp2)))

        diff = snr1 - snr2
        print(f'{freq:>8}  {name:>8}  {m:>4}  {snr1:>14.2f}  {snr2:>14.2f}  {diff:>+10.2f}')
    print()